# Lecture 12: Markov Decision Processes — Exercises

Companion to the note [Noter små](../Noter%20små.md).

Practise planning under uncertainty with the adventurer–gold–snake grid world from the lecture: formalising MDPs, discounted utilities, transition probabilities, Bellman updates and value iteration by hand (exam style), policy evaluation as a linear system, and value and policy iteration in NumPy.

**15 exercises** · 🧠 Concept ×3 · ✍️ By hand ×6 · 💻 Code ×6

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

# The grid world from the lecture. Cells are (column, row), (1,1) is the north-west corner.
#   gold at (3,1) (terminal), snake at (3,2). A move succeeds with probability 0.8;
#   with probability 0.2 the adventurer slips to the cell to the RIGHT (east) of where she stands.
#   Moving into the wall means staying in place.
cells = [(c, r) for r in (1, 2, 3) for c in (1, 2, 3)]      # state index = position in this list
idx = {s: i for i, s in enumerate(cells)}
ACTIONS = ['N', 'E', 'S', 'W']
_DIR = {'N': (0, -1), 'E': (1, 0), 'S': (0, 1), 'W': (-1, 0)}

def _move(s, d):
    c, r = s[0] + d[0], s[1] + d[1]
    return (c, r) if 1 <= c <= 3 and 1 <= r <= 3 else s

P = np.zeros((4, 9, 9))          # P[a, s, s'] = P(s' | s, a)
for _a, _name in enumerate(ACTIONS):
    for _s in cells:
        P[_a, idx[_s], idx[_move(_s, _DIR[_name])]] += 0.8
        P[_a, idx[_s], idx[_move(_s, (1, 0))]] += 0.2

terminal = np.zeros(9, bool); terminal[idx[(3, 1)]] = True
def make_R(gold=10.0, snake=-5.0, other=-0.1):
    R = np.full(9, other); R[idx[(3, 1)]] = gold; R[idx[(3, 2)]] = snake
    return R
R = make_R()          # lecture rewards: gold +10, snake -5, elsewhere -0.1
gamma = 0.9
start = idx[(1, 3)]   # we let the adventurer start in the south-west corner

def show(V):
    """Print a value function (length-9 array) as a 3x3 grid (rows = row 1..3)."""
    print(np.round(np.asarray(V, float).reshape(3, 3), 3))

def show_policy(pi):
    arrows = {0: '↑', 1: '→', 2: '↓', 3: '←'}
    for r in range(3):
        print(' '.join('G' if terminal[3 * r + c] else arrows[int(pi[3 * r + c])] for c in range(3)))

## Part A · Concepts

### Exercise 1 · Is it an MDP?
*🧠 Concept · ★☆☆*

1. List the three properties of the world that the note says an MDP assumes.
2. Suppose the adventurer only gets the +10 the **first** time she reaches the gold, and the gold square is no longer terminal. Why does the
   model with states = positions violate the Markov assumption, and how do you repair it?
3. Why is the solution of an MDP a *policy* (a map from states to actions) rather than a fixed sequence of actions as in classical planning?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

The Markov property: the next state and reward depend only on the current state and action.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. The world is **fully observable**; actions are **non-deterministic** with transition probabilities that do not depend on time; the
   **reward** depends on the current state (not on the time step).
2. The reward in the gold square would depend on the history (been there before or not), not just on the current position. Add a state
   variable *hasGold* ∈ {true, false}: states become (position, hasGold), and the reward depends only on the current state again.
3. Because outcomes are random, a fixed action sequence may lead anywhere. A policy says what to do in **every** state you might end up in
   (a plan that is "closed" for all reachable states).

</details>

### Exercise 2 · Finite horizon or discounting?
*🧠 Concept · ★☆☆*

1. Why can the plain sum of rewards over an infinite horizon be useless for comparing policies?
2. Explain the two fixes from the note and the behaviour of a finite-horizon agent with $k = 0$.
3. Give two interpretations of the discount factor $\gamma$. What happens for $\gamma = 0$ and $\gamma \to 1$?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Look at the infinite sum of $-0.1$ per step, or of $+1$ per step.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. The sum can be $\pm\infty$ for many policies, so they cannot be ranked (every policy that keeps collecting $+1$ is "equally infinite").
2. *Finite horizon*: only count the first $k$ rewards. With $k = 0$ the agent is completely greedy (only the immediate reward), and in general
   it may walk into a bad state just after the horizon ("live as if there is no tomorrow"). *Discounting*: $\sum_i\gamma^iR(s_i)$ with
   $\gamma < 1$, which is bounded by $R_{\max}/(1-\gamma)$.
3. (a) Inflation/interest rate: a reward now is worth more than later. (b) With probability $1-\gamma$ the process terminates at each step
   (e.g. the robot breaks down). $\gamma = 0$: greedy; $\gamma \to 1$: far-sighted, approaching additive rewards. Typical choice $\gamma = 0.99$.

</details>

### Exercise 3 · Formalise a game as an MDP
*🧠 Concept · ★★☆*

Formalise the game **2048** (4×4 board of tiles; the player slides all tiles up/down/left/right, equal tiles merge into their sum,
then a new tile, 2 with probability 0.9 or 4 with probability 0.1, appears on a uniformly random empty cell) as an MDP:
states, actions, reward function, transition function, initial state(s) and terminal states. Is the transition function deterministic?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Separate the deterministic part (sliding and merging) from the random part (the new tile).

</details>

<details>
<summary><b>✅ Solution</b></summary>

* **States**: the content of the 16 cells (empty or a power of two), optionally plus the score.
* **Actions**: {up, down, left, right} (an action that moves nothing can be disallowed or treated as a no-op).
* **Reward**: the score gained by the move, i.e. the sum of the merged tiles (alternatively $+1$ per survived move).
* **Transitions**: sliding/merging is deterministic; then a 2 (prob. $0.9$) or 4 (prob. $0.1$) is placed on each of the $m$ empty cells
  with probability $1/m$. So $P(s'\mid s,a) = \frac{0.9}{m}$ or $\frac{0.1}{m}$ for the reachable boards, 0 otherwise: **not deterministic**.
* **Initial states**: an empty board with two random tiles (a distribution over initial states).
* **Terminal states**: boards where no move changes anything (game over), optionally also boards containing 2048.

</details>

## Part B · By hand

### Exercise 4 · Discounted utility of a run
*✍️ By hand · ★☆☆*

The adventurer visits states with rewards $-0.1, -0.1, -0.1, +10$ (then the episode ends).

1. Compute $U = \sum_i\gamma^iR(s_i)$ for $\gamma = 0.9$ and for $\gamma = 0.5$.
2. Give an upper bound on the discounted utility of **any** infinite run when $R_{\max} = 10$ and $\gamma = 0.9$.

In [ ]:
U_09 = None
U_05 = None
U_bound = None

_r = np.array([-0.1, -0.1, -0.1, 10])
check('U (gamma=0.9)', U_09, np.sum(0.9 ** np.arange(4) * _r)); check('U (gamma=0.5)', U_05, np.sum(0.5 ** np.arange(4) * _r))
check('bound', U_bound, 10 / (1 - 0.9))

<details>
<summary><b>💡 Hint</b></summary>

Geometric series: $\sum_{i\ge0}\gamma^i = \frac{1}{1-\gamma}$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. $\gamma = 0.9$: $-0.1 - 0.09 - 0.081 + 7.29 = 7.019$. $\gamma = 0.5$: $-0.1 - 0.05 - 0.025 + 1.25 = 1.075$.
2. $\sum_i\gamma^iR(s_i) \le R_{\max}\sum_i\gamma^i = \frac{10}{1 - 0.9} = 100$.

```python
U_09 = 7.019
U_05 = 1.075
U_bound = 100.0
```

</details>

### Exercise 5 · Reading off transition probabilities
*✍️ By hand · ★☆☆*

In the grid world (move succeeds with prob. 0.8, otherwise slip to the cell east of the current one; walls mean staying put), give the
distributions over next cells for

1. action **north** in $(1,1)$,
2. action **east** in $(2,2)$,
3. action **west** in $(3,3)$.

Store them as dicts `{(col, row): probability}`.

In [ ]:
out_north_11 = None
out_east_22 = None
out_west_33 = None

def _dist(a, s):
    row = P[ACTIONS.index(a), idx[s]]
    return {cells[j]: row[j] for j in np.nonzero(row)[0]}
for _name, _got, _a, _s in [('north in (1,1)', out_north_11, 'N', (1, 1)), ('east in (2,2)', out_east_22, 'E', (2, 2)),
                            ('west in (3,3)', out_west_33, 'W', (3, 3))]:
    _ref = _dist(_a, _s)
    check(_name, None if _got is None else (set(_got) == set(_ref) and all(abs(_got[k] - _ref[k]) < 1e-9 for k in _ref)), True)

<details>
<summary><b>💡 Hint</b></summary>

Outcomes that land in the same cell add up.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. North from $(1,1)$ hits the wall: stay with 0.8; slip east to $(2,1)$ with 0.2. → `{(1,1): 0.8, (2,1): 0.2}`.
2. East from $(2,2)$: intended $(3,2)$ with 0.8 and slipping east also gives $(3,2)$ → `{(3,2): 1.0}`: straight into the snake!
3. West from $(3,3)$: $(2,3)$ with 0.8; slipping east hits the wall, so stay with 0.2 → `{(2,3): 0.8, (3,3): 0.2}`.

```python
out_north_11 = {(1, 1): 0.8, (2, 1): 0.2}
out_east_22 = {(3, 2): 1.0}
out_west_33 = {(2, 3): 0.8, (3, 3): 0.2}
```

</details>

### Exercise 6 · One Bellman update
*✍️ By hand · ★★☆*

Lecture rewards ($+10$ gold, $-5$ snake, $-0.1$ elsewhere), $\gamma = 0.9$. Suppose the current value estimate is

| | col 1 | col 2 | col 3 |
|:--:|:--:|:--:|:--:|
| row 1 | 6 | 8 | 10 |
| row 2 | 5 | 4 | 1 |
| row 3 | 4 | 3 | 2 |

Apply one Bellman update $V(s) \leftarrow R(s) + \max_a\gamma\sum_{s'}P(s'\mid s,a)V(s')$ to the state $(2,2)$: give the value of each action's
sum $\sum_{s'}P(s'\mid s,a)V(s')$, the best action and the new $V(2,2)$.

In [ ]:
q_sums = None    # dict {'N': ..., 'E': ..., 'S': ..., 'W': ...}
best_a = None
V22_new = None

_V = np.array([6, 8, 10, 5, 4, 1, 4, 3, 2.]); _s = idx[(2, 2)]; _q = P[:, _s] @ _V
check('expected next values', None if q_sums is None else [q_sums[a] for a in ACTIONS], _q)
check('best action', best_a, ACTIONS[int(np.argmax(_q))]); check('new V(2,2)', V22_new, R[_s] + gamma * _q.max())

<details>
<summary><b>💡 Hint</b></summary>

Every action has the 0.2 slip to $(3,2)$ (value 1), except east where both outcomes are $(3,2)$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

* N: $0.8\cdot V(2,1) + 0.2\cdot V(3,2) = 0.8\cdot8 + 0.2\cdot1 = 6.6$
* E: $1.0\cdot V(3,2) = 1$
* S: $0.8\cdot3 + 0.2\cdot1 = 2.6$
* W: $0.8\cdot5 + 0.2\cdot1 = 4.2$

Best action **N**; $V(2,2) \leftarrow -0.1 + 0.9\cdot6.6 = 5.84$.

```python
q_sums = {'N': 6.6, 'E': 1.0, 'S': 2.6, 'W': 4.2}
best_a = 'N'
V22_new = 5.84
```

</details>

### Exercise 7 · Three iterations of value iteration (exam variant)
*✍️ By hand · ★★★*

Exercise-sheet variant: the gold gives $+1$, the snake $-20$, everything else $0$; $\gamma = 0.9$; the gold square is terminal ($V = R$ there).
Start from $V_0 = 0$ and compute $V_1$, $V_2$, $V_3$. Store $V_3$ for the cells $(1,1)$, $(1,2)$, $(2,2)$, $(3,2)$.
Looking at $V_3$, what will an adventurer starting in the bottom row do?

In [ ]:
V3_11 = None
V3_12 = None
V3_22 = None
V3_32 = None

_Rv = make_R(1, -20, 0); _V = np.zeros(9)
for _ in range(3):
    _V = np.where(terminal, _Rv, _Rv + 0.9 * (P @ _V).max(0))
check('V3(1,1)', V3_11, _V[idx[(1, 1)]], tol=1e-3); check('V3(1,2)', V3_12, _V[idx[(1, 2)]], tol=1e-3)
check('V3(2,2)', V3_22, _V[idx[(2, 2)]], tol=1e-3); check('V3(3,2)', V3_32, _V[idx[(3, 2)]], tol=1e-3)

<details>
<summary><b>💡 Hint 1</b></summary>

$V_1 = R$. In iteration 2 only cells next to the gold or the snake change.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

$V_2(3,2) = -20 + 0.9\cdot(0.8\cdot1 + 0.2\cdot(-20)) = -22.88$ (north towards the gold; the slip east hits the wall).

</details>

<details>
<summary><b>✅ Solution</b></summary>

$V_1$: gold 1, snake $-20$, all others 0.

$V_2$: $(2,1)$: east gives $0.9\cdot1 = 0.9$. $(2,2)$: every action has a 0.2 slip into the snake, best is $0.9\cdot0.2\cdot(-20) = -3.6$.
$(3,2)$: $-22.88$ (hint). Others stay 0.

$V_3$: $(1,1)$: east: $0.9\cdot(0.8\cdot0.9 + 0.2\cdot0.9) = 0.81$. $(1,2)$: every move slips into $(2,2)$ with 0.2: $0.9\cdot0.2\cdot(-3.6) = -0.648$.
$(2,2)$: north: $0.9\cdot(0.8\cdot0.9 + 0.2\cdot(-22.88)) = -3.470$. $(3,2)$: $-20 + 0.9\cdot(0.8\cdot1 + 0.2\cdot(-22.88)) = -23.398$.

The bottom row keeps value 0 and every route upwards passes cells with negative value, so an adventurer starting at the bottom stays there
and never goes for the gold: with a $-20$ snake and only $+1$ gold, the risk is not worth it.

```python
V3_11 = 0.81
V3_12 = -0.648
V3_22 = -3.4704
V3_32 = -23.3984
```

</details>

### Exercise 8 · Policy evaluation as a linear system
*✍️ By hand · ★★☆*

A two-state MDP with a fixed policy $\pi$: $R(A) = 0$, $R(B) = 1$, $\gamma = 0.9$. Under $\pi$, from $A$ you move to $B$ with probability $0.5$
(else stay in $A$); from $B$ you stay in $B$ with probability $0.8$ (else go to $A$).

Write the two equations $U_\pi(s) = R(s) + \gamma\sum_{s'}P(s'\mid\pi(s),s)U_\pi(s')$ and solve them.

In [ ]:
U_A = None
U_B = None

_T = np.array([[0.5, 0.5], [0.2, 0.8]]); _U = np.linalg.solve(np.eye(2) - 0.9 * _T, [0., 1.])
check('U(A)', U_A, _U[0], tol=1e-3); check('U(B)', U_B, _U[1], tol=1e-3)

<details>
<summary><b>💡 Hint</b></summary>

From the first equation, $U_A(1 - 0.45) = 0.45\,U_B$, so $U_A = \frac{9}{11}U_B$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$U_A = 0.9(0.5U_A + 0.5U_B) \Rightarrow U_A = \tfrac{9}{11}U_B$.
$U_B = 1 + 0.9(0.8U_B + 0.2U_A) = 1 + 0.72U_B + 0.18\cdot\tfrac{9}{11}U_B \Rightarrow U_B(0.28 - 0.14727) = 1 \Rightarrow U_B \approx 7.534$, $U_A \approx 6.164$.

In matrix form $U = R + \gamma T_\pi U \Leftrightarrow (I - \gamma T_\pi)U = R$: this is the evaluation step of policy iteration.

```python
U_B = 1 / (0.28 - 0.18 * 9 / 11)
U_A = 9 / 11 * U_B
```

</details>

### Exercise 9 · How many iterations are enough?
*✍️ By hand · ★★☆*

The Bellman operator is a $\gamma$-contraction in the max-norm: $\lVert BV - BV'\rVert_\infty \le \gamma\lVert V - V'\rVert_\infty$.
Hence $\lVert V_k - V^*\rVert_\infty \le \gamma^k\lVert V_0 - V^*\rVert_\infty$.

With $\gamma = 0.9$ and an initial error of at most 10, what is the smallest $k$ that guarantees an error below $0.01$? And with $\gamma = 0.99$?

In [ ]:
k_09 = None
k_099 = None

check('k for gamma=0.9', k_09, int(np.ceil(np.log(0.01 / 10) / np.log(0.9))))
check('k for gamma=0.99', k_099, int(np.ceil(np.log(0.01 / 10) / np.log(0.99))))

<details>
<summary><b>💡 Hint</b></summary>

Solve $10\,\gamma^k < 0.01$ for $k$ using logarithms (careful: $\log\gamma < 0$ flips the inequality).

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\gamma^k < 10^{-3} \Leftrightarrow k > \frac{\ln 10^{-3}}{\ln\gamma}$. For $\gamma = 0.9$: $k > 65.6$, so $k = 66$. For $\gamma = 0.99$: $k > 687.3$, so $k = 688$.
Far-sighted agents are more expensive to plan for: value iteration converges about ten times slower.

```python
k_09 = 66
k_099 = 688
```

</details>

## Part C · Code

### Exercise 10 · Value iteration
*💻 Code · ★★☆*

Implement `value_iteration(P, R, gamma, terminal, tol)` that starts from $V = 0$ and repeatedly applies the Bellman operator
$BV(s) = R(s) + \max_a\gamma\sum_{s'}P(s'\mid s,a)V(s')$ (with $BV(s) = R(s)$ for terminal states) until the largest change is below `tol`.
Return `(V, n_iterations)`. Run it on the lecture rewards and print the grid with `show(V_star)`.

In [ ]:
def value_iteration(P, R, gamma, terminal, tol=1e-10):
    # TODO
    return None

res_vi = value_iteration(P, R, gamma, terminal)
V_star = None if res_vi is None else res_vi[0]

if V_star is not None:
    _BV = np.where(terminal, R, R + gamma * (P @ V_star).max(0))
    check('V* is a fixed point of the Bellman operator', np.abs(_BV - V_star).max() < 1e-8, True)
    _pi = (P @ V_star).argmax(0); _T = P[_pi, np.arange(9)]; _T[terminal] = 0
    check('V* equals the value of its greedy policy', V_star, np.linalg.solve(np.eye(9) - gamma * _T, R), tol=1e-6)
else:
    check('V* is a fixed point of the Bellman operator', None, True)

<details>
<summary><b>💡 Hint</b></summary>

`P @ V` has shape (4, 9): entry `[a, s]` is $\sum_{s'}P(s'\mid s,a)V(s')$. Then `.max(0)` maximises over actions.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$V^*$ is about 10 next to the gold and decreases with distance; $(3,2)$ (the snake) is about 2.68. The check verifies two independent
facts: $V^*$ satisfies the Bellman equation, and it equals the value of the policy that is greedy with respect to it.

```python
def value_iteration(P, R, gamma, terminal, tol=1e-10):
    V = np.zeros(len(R)); n = 0
    while True:
        V_new = np.where(terminal, R, R + gamma * (P @ V).max(0)); n += 1
        if np.abs(V_new - V).max() < tol:
            return V_new, n
        V = V_new

res_vi = value_iteration(P, R, gamma, terminal)
V_star = res_vi[0]
show(V_star); print('iterations:', res_vi[1])
```

</details>

### Exercise 11 · Extracting the optimal policy
*💻 Code · ★☆☆*

Implement `greedy_policy(P, V)` returning, for every state, the index of the action maximising $\sum_{s'}P(s'\mid s,a)V(s')$.
Print it with `show_policy`. Which route does the adventurer take from the south-west corner $(1,3)$? Why does she climb column 1 instead of column 2?

In [ ]:
def greedy_policy(P, V):
    # TODO
    return None

pi_star = None if V_star is None else greedy_policy(P, V_star)

if pi_star is not None:
    _T = P[np.asarray(pi_star), np.arange(9)].copy(); _T[terminal] = 0
    check('greedy policy achieves V*', np.linalg.solve(np.eye(9) - gamma * _T, R), V_star, tol=1e-6)
else:
    check('greedy policy achieves V*', None, True)

<details>
<summary><b>💡 Hint</b></summary>

`np.argmax(P @ V, axis=0)`. $R(s)$ and $\gamma$ do not change the argmax.

</details>

<details>
<summary><b>✅ Solution</b></summary>

```
→ → G
↑ ↑ ↑
↑ ↑ ←
```
From $(1,3)$ she goes north twice along column 1, then east along row 1. Every move carries a 0.2 slip east; from $(2,2)$ that slip lands
on the snake, whereas from column 1 a slip only lands in column 2. So column 1 is the safer way up. The snake cell itself points north
(escape towards the gold), and $(3,3)$ points west because going north would enter the snake.

```python
def greedy_policy(P, V):
    return np.argmax(P @ V, axis=0)

pi_star = greedy_policy(P, V_star)
show_policy(pi_star)
```

</details>

### Exercise 12 · Exact policy evaluation
*💻 Code · ★★☆*

Implement `evaluate_policy(P, R, gamma, terminal, pi)` that solves the linear system $U = R + \gamma T_\pi U$ (with zero future value in
terminal states) using `np.linalg.solve`. Evaluate the silly policy "always go east" and print it.

In [ ]:
def evaluate_policy(P, R, gamma, terminal, pi):
    # TODO
    return None

U_east = evaluate_policy(P, R, gamma, terminal, np.full(9, ACTIONS.index('E')))

_pi = np.full(9, 1); _U = np.zeros(9)
for _ in range(3000):
    _U = np.where(terminal, R, R + gamma * (P[_pi, np.arange(9)] @ _U))
check('matches iterative evaluation', U_east, _U, tol=1e-6)

<details>
<summary><b>💡 Hint</b></summary>

`T = P[pi, np.arange(9)]` picks row `P[pi[s], s]` for every state. Zero the rows of terminal states.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Always going east sends everybody in row 2 straight into the snake, where east keeps her in place: $U(3,2) = -5/(1 - 0.9) = -50$.
Bottom-row cells end up in $(3,3)$ forever: $-0.1/(1-0.9) = -1$. Only row 1 reaches the gold. Solving the linear system costs
$O(|S|^3)$ but needs no iterations.

```python
def evaluate_policy(P, R, gamma, terminal, pi):
    n = len(R)
    T = P[np.asarray(pi), np.arange(n)].copy()
    T[terminal] = 0
    return np.linalg.solve(np.eye(n) - gamma * T, R)

U_east = evaluate_policy(P, R, gamma, terminal, np.full(9, ACTIONS.index('E')))
show(U_east)
```

</details>

### Exercise 13 · Policy iteration
*💻 Code · ★★☆*

Implement policy iteration as in the note: start from "always north", then alternate **evaluation** (your `evaluate_policy`) and
**improvement** ($\pi_{i+1}(s) = \arg\max_a\sum_{s'}P(s'\mid s,a)U_{\pi_i}(s')$) until the policy no longer changes.
Return `(pi, U, n_iterations)` and compare with value iteration.

In [ ]:
def policy_iteration(P, R, gamma, terminal):
    # TODO
    return None

res_pi = policy_iteration(P, R, gamma, terminal)

check('U equals V* from value iteration', None if res_pi is None else res_pi[1], value_iteration(P, R, gamma, terminal)[0] if value_iteration(P, R, gamma, terminal) is not None else None, tol=1e-6)

<details>
<summary><b>💡 Hint</b></summary>

Stop when `np.array_equal(pi_new, pi)`. To avoid cycling between equally good actions, only change the action when it is strictly better.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Policy iteration converges in very few iterations (here 3–4), each of which is more expensive (a linear solve) than a value-iteration
sweep. Both give the same $V^*$ and policy.

```python
def policy_iteration(P, R, gamma, terminal):
    pi = np.zeros(len(R), int)
    n = 0
    while True:
        U = evaluate_policy(P, R, gamma, terminal, pi); n += 1
        Q = P @ U
        best = Q.argmax(0)
        improve = Q[best, np.arange(len(R))] > Q[pi, np.arange(len(R))] + 1e-12
        pi_new = np.where(improve, best, pi)
        if np.array_equal(pi_new, pi):
            return pi, U, n
        pi = pi_new

res_pi = policy_iteration(P, R, gamma, terminal)
show_policy(res_pi[0]); print('iterations:', res_pi[2])
```

</details>

### Exercise 14 · Rewards shape behaviour
*💻 Code · ★★☆*

Solve the exam variant (gold $+1$, snake $-20$, other $0$, $\gamma = 0.9$) to convergence. Store $V^*$ in `V_variant` and the greedy policy in
`pi_variant`. Compare the action in the start cell $(1,3)$ and the value $V^*(1,3)$ with the lecture rewards. Explain.

In [ ]:
V_variant = None
pi_variant = None

_Rv = make_R(1, -20, 0); _V = np.zeros(9)
for _ in range(2000):
    _V = np.where(terminal, _Rv, _Rv + 0.9 * (P @ _V).max(0))
check('V* of the variant', V_variant, _V, tol=1e-6)
check('the start cell no longer heads north', None if pi_variant is None else ACTIONS[int(pi_variant[start])] != 'N', True)

<details>
<summary><b>💡 Hint</b></summary>

`make_R(1, -20, 0)` builds the reward vector.

</details>

<details>
<summary><b>✅ Solution</b></summary>

With the lecture rewards, $(1,3)$ goes north and $V^*(1,3) \approx 5.83$. In the variant, $V^*(1,3) = 0$ and the policy stays in the bottom
row (e.g. heads east along it, where nothing bad can happen): the bottom row has value 0, while any path north passes cells that risk a
$-20$ snake for at most $+1$. Same dynamics, different rewards: the reward function **is** the specification of the task.

```python
V_variant = value_iteration(P, make_R(1, -20, 0), 0.9, terminal)[0]
pi_variant = greedy_policy(P, V_variant)
show(V_variant); show_policy(pi_variant)
```

</details>

### Exercise 15 · The discount factor shapes behaviour
*💻 Code · ★★★*

With the lecture rewards, solve the MDP for $\gamma \in \{0.1, 0.5, 0.9, 0.99\}$. Store $V^*(1,3)$ in a dict `v_start` and the optimal action
(a letter) in cell $(2,3)$ in a dict `act_23`. Explain why the action in $(2,3)$ changes twice.

In [ ]:
v_start = None
act_23 = None

_res_v, _res_a = {}, {}
for _g in (0.1, 0.5, 0.9, 0.99):
    _V = np.zeros(9)
    for _ in range(5000):
        _V = np.where(terminal, R, R + _g * (P @ _V).max(0))
    _res_v[_g] = _V[start]; _res_a[_g] = ACTIONS[int((P @ _V)[:, idx[(2, 3)]].argmax())]
check('V*(1,3) per gamma', None if v_start is None else [v_start[g] for g in _res_v], list(_res_v.values()), tol=1e-4)
check('action in (2,3) per gamma', None if act_23 is None else [act_23[g] for g in _res_a], list(_res_a.values()))

<details>
<summary><b>💡 Hint</b></summary>

Reuse `value_iteration` and `greedy_policy`; `ACTIONS[a]` turns an index into a letter.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$V^*(1,3)$ grows with $\gamma$ (≈ −0.11, 0.37, 5.83, 8.85): the distant gold counts for more. In $(2,3)$:

* $\gamma = 0.1$: **W**. The gold is worth almost nothing from there ($0.1^3\cdot10$), so the agent just avoids moving towards the snake's neighbourhood.
* $\gamma = 0.5, 0.9$: **N**, the short route through $(2,2)$, accepting the 0.2 risk of slipping into the snake.
* $\gamma = 0.99$: **W** again, now for the opposite reason: the agent is patient enough to take the longer, safe route through column 1,
  because a few extra $-0.1$ steps cost less than the snake risk when the gold is barely discounted.

```python
v_start, act_23 = {}, {}
for g in (0.1, 0.5, 0.9, 0.99):
    V = value_iteration(P, R, g, terminal)[0]
    v_start[g] = V[start]
    act_23[g] = ACTIONS[int(greedy_policy(P, V)[idx[(2, 3)]])]
print(v_start, act_23)
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [Noter små](../Noter%20små.md).*